# 04 — MICM Fuel Prices | Bronze Ingestion

**Project:** RD Economic & Financial Intelligence Platform

Ingest the official MICM weekly fuel price file (Dominican open-data portal) into Bronze, preserving source values.

| Item | Value |
|---|---|
| Files | `precios-de-combustibles-2010-2026.xlsx` (authoritative, decimal precision) and `.csv` (same data rounded to whole pesos, kept for reconciliation) |
| Raw landing | `/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/` |
| Target | `rd_economic_intelligence.bronze.micm_fuel_prices_raw` (one row per source row and format) |

**Source decision:** profiling showed the CSV equals the XLSX rounded to whole pesos on all 872 rows (no difference above 0.5). The XLSX keeps the published decimals, so it is the authoritative source; the CSV is ingested only to prove that reconciliation on every run.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import uuid, json
import pandas as pd
from pyspark.sql import functions as F

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/micm_raw/fuel_prices/"
TARGET = f"{CATALOG}.bronze.micm_fuel_prices_raw"
FILES = {"xlsx": "precios-de-combustibles-2010-2026.xlsx", "csv": "precios-de-combustibles-2010-2026.csv"}
REQUIRED = ["DIA DESDE", "DIA HASTA", "MES", "AÑO", "GASOLINA PREMIUM", "GASOLINA REGULAR", "GASOIL OPTIMO", "GLP"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

landed = {f.name: f for f in dbutils.fs.ls(RAW_PATH)}
missing = [f for f in FILES.values() if f not in landed]
if missing:
    raise FileNotFoundError(f"Missing in {RAW_PATH}: {missing}")

In [0]:
# Read both formats; keep every source column, values as published (text in Bronze)
frames = []
for fmt, name in FILES.items():
    path = RAW_PATH + name
    pdf = pd.read_excel(path, dtype=object) if fmt == "xlsx" else pd.read_csv(path, sep=";", encoding="latin-1", dtype=str)
    pdf.columns = [str(c).strip() for c in pdf.columns]
    absent = [c for c in REQUIRED if c not in pdf.columns]
    if absent:
        raise ValueError(f"{name}: layout changed, missing columns {absent}")
    source_columns = list(pdf.columns)
    for i, row in enumerate(pdf.itertuples(index=False), start=2):
        frames.append((fmt, name, i, json.dumps({c: (None if pd.isna(v) else str(v)) for c, v in zip(source_columns, row)}, ensure_ascii=False),
                       int(landed[name].size), datetime.fromtimestamp(landed[name].modificationTime / 1000, tz=timezone.utc)))
    print(f"{fmt}: {len(pdf):,} rows, {len(source_columns)} columns")

df = (spark.createDataFrame(frames, "_source_format string, _source_file string, _source_row_number int, record_json string, _source_file_size_bytes bigint, _source_modified_ts timestamp")
      .withColumn("_ingestion_ts", F.lit(INGESTION_TS).cast("timestamp")).withColumn("_batch_id", F.lit(BATCH_ID)))

In [0]:
if spark.catalog.tableExists(TARGET):
    spark.sql(f"DELETE FROM {TARGET} WHERE _source_file IN ({', '.join(repr(f) for f in FILES.values())})")
df.write.format("delta").mode("append").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: MICM weekly fuel prices as published (XLSX authoritative, CSV for reconciliation). One row per source row; all source columns kept in record_json.'")

loaded = spark.table(TARGET).filter(F.col("_batch_id") == BATCH_ID).groupBy("_source_format").count().collect()
counts = {r["_source_format"]: r["count"] for r in loaded}
assert sum(counts.values()) == len(frames), "Row count mismatch"
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "batch_id": BATCH_ID, "rows": counts}))